In [9]:
import rasterio
import numpy as np
import os

def normalize_to_255(input_path, output_path):
    with rasterio.open(input_path) as src:
        band = src.read(1).astype(np.float32)
        profile = src.profile.copy()

        nodata = src.nodata
        mask = np.ones_like(band, dtype=bool)
        
        if nodata is not None:
            mask &= (band != nodata)
        mask &= ~np.isnan(band)  # Exclude NaNs explicitly

        valid_pixels = band[mask]
        if valid_pixels.size == 0:
            print("No valid pixels to normalize.")
            return

        min_val = np.nanmin(valid_pixels)
        max_val = np.nanmax(valid_pixels)

        normed_band = np.zeros_like(band, dtype=np.uint8)

        if max_val != min_val:
            scaled = (band[mask] - min_val) / (max_val - min_val)
            scaled = np.clip(scaled, 0, 1)  # Ensure within [0, 1]
            normed_band[mask] = (scaled * 255).astype(np.uint8)

        # Update profile
        profile.update(
            dtype='uint8',
            count=1,
            compress='lzw'
        )
        if nodata is not None:
            profile.update(nodata=0)
        else:
            profile.pop('nodata', None)

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(normed_band, 1)

        print(f"Normalized raster saved to {output_path}")



if __name__ == "__main__":
    input_file = "/home/ec2-user/SageMaker/ValidationDataset/Upper_Willow_Creek_BareEarth_DEM_1m_1.tif"
    output_file = "/home/ec2-user/SageMaker/ValidationDataset/NormalizedDEM.tif"
    normalize_to_255(input_file, output_file)


Normalized raster saved to /home/ec2-user/SageMaker/ValidationDataset/NormalizedDEM.tif
